## Extract the zipfile

In [23]:
import zipfile

zip_path = "/content/idflakies-output.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content/idflakies-output")

In [24]:
import os
import re
import json
import glob
import pandas as pd

base_path = "/content/idflakies-output/all-output/kevinsawicki.http-request_output"

# Information gained from previous runs
repo_url = "https://github.com/kevinsawicki/http-request"
commit_sha = "2d62a3e9da726942a93cf16b6e91c0187e6c0136"

# Load the flaky test CSV
flaky_csv = glob.glob(
    "/content/idflakies-output/**/all_flaky_tests_list.csv",
    recursive=True
)[0]

df_flaky = pd.read_csv(flaky_csv, header=None)

df_flaky.columns = [
    "test_name",
    "detection_result_path"
]

print("Row count:", len(df_flaky))
print("Unique test count:", df_flaky["test_name"].nunique())

df_flaky.head()

Row count: 80
Unique test count: 16


,test_name,detection_result_path
0,com.github.kevinsawicki.http.HttpRequestTest.b...,./lib/detection-results/random-class/round1.json
1,com.github.kevinsawicki.http.HttpRequestTest.b...,./lib/detection-results/random-class/round3.json
2,com.github.kevinsawicki.http.HttpRequestTest.b...,./lib/detection-results/random/round1.json
3,com.github.kevinsawicki.http.HttpRequestTest.b...,./lib/detection-results/random/round3.json
4,com.github.kevinsawicki.http.HttpRequestTest.b...,./lib/detection-results/reverse/round0.json


In [34]:
# Extract information
records = []

for _, row in df_flaky.iterrows():
    test_name = row["test_name"]
    path = row["detection_result_path"]

    # Get the detector name from path
    detector_type = path.split("/")[3]

    # Get round number
    round_n = int(re.search(r"round(\d+)", path).group(1))

    # Open json path
    json_path = os.path.join(
        base_path,
        path.replace("./", "")
    )

    with open(json_path, "r") as f:
        data = json.load(f)

    # Get detected tests (dts is dependent tests)
    dts = data["filteredTests"]["dts"]

    # Find the matching test (connects test in the CSV file to the json record)
    match_found = None

    for dt in dts:
        if dt["name"] == test_name:
            match_found = dt
            break

    # If not found in filtered tests skip and go on to the next
    if match_found is None:
        continue

    intended = match_found["intended"]
    revealed = match_found["revealed"]

    intended_order = intended["order"]
    revealed_order = revealed["order"]

    records.append({
        "repo_url": repo_url,
        "commit_sha": commit_sha,
        "test_name": test_name,

        "detector_type": detector_type,
        "round": round_n,

        "flaky_type": match_found["type"],

        "intended_result": intended["result"],
        "revealed_result": revealed["result"],

        "intended_length": len(intended_order),  # length of order that gave intended result
        "revealed_length": len(revealed_order), # length of order that gave different results
        # difference in length
        "test_length_diff": abs(
            len(revealed_order) - len(intended_order)),

        "round_time": data["roundTime"],
        "num_test_runs": len(data["testRunIds"])
    })

In [35]:
detect_df = pd.DataFrame(records)
detect_df.head(10)

,repo_url,commit_sha,test_name,detector_type,round,flaky_type,intended_result,revealed_result,intended_length,revealed_length,test_length_diff,round_time,num_test_runs
0,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.b...,random-class,1,OD,PASS,ERROR,27,135,108,27.547374,1
1,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.b...,random,1,OD,PASS,ERROR,27,135,108,27.585113,1
2,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.b...,reverse,0,OD,PASS,ERROR,27,135,108,27.514514,1
3,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.d...,random-class,1,OD,ERROR,PASS,38,124,86,27.547374,1
4,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.d...,random,1,OD,ERROR,PASS,38,124,86,27.585113,1
5,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.d...,reverse,0,OD,ERROR,PASS,38,124,86,27.514514,1
6,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.d...,random-class,1,OD,ERROR,PASS,39,123,84,27.547374,1
7,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.d...,random,1,OD,ERROR,PASS,39,123,84,27.585113,1
8,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.d...,reverse,0,OD,ERROR,PASS,39,123,84,27.514514,1
9,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.d...,random-class,1,OD,ERROR,PASS,40,122,82,27.547374,1


In [39]:
final_df = (
    detect_df
    .groupby(["repo_url", "commit_sha", "test_name", "flaky_type"], as_index=False)
    .agg(
        total_detection_count=("test_name", "size"),
        unique_detector_count=("detector_type", "nunique"),
        avg_intended_length=("intended_length", "mean"),
        avg_revealed_length=("revealed_length", "mean"),
        avg_length_diff=("test_length_diff", "mean"),
        avg_round_time=("round_time", "mean"),
        avg_num_test_runs=("num_test_runs", "mean")
    )
)

final_df.head(10)

,repo_url,commit_sha,test_name,flaky_type,total_detection_count,unique_detector_count,avg_intended_length,avg_revealed_length,avg_length_diff,avg_round_time,avg_num_test_runs
0,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.b...,OD,3,3,27.0,135.0,108.0,27.549,1.0
1,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.d...,OD,3,3,38.0,124.0,86.0,27.549,1.0
2,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.d...,OD,3,3,39.0,123.0,84.0,27.549,1.0
3,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.d...,OD,3,3,40.0,122.0,82.0,27.549,1.0
4,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.d...,OD,3,3,41.0,121.0,80.0,27.549,1.0
5,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.g...,OD,3,3,73.0,89.0,16.0,27.549,1.0
6,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.g...,OD,3,3,74.0,88.0,14.0,27.549,1.0
7,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.g...,OD,3,3,75.0,87.0,12.0,27.549,1.0
8,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.g...,OD,3,3,76.0,86.0,10.0,27.549,1.0
9,https://github.com/kevinsawicki/http-request,2d62a3e9da726942a93cf16b6e91c0187e6c0136,com.github.kevinsawicki.http.HttpRequestTest.g...,OD,3,3,77.0,85.0,8.0,27.549,1.0


## Save to new csv file

In [40]:
final_df.to_csv("/content/flaky_sample_dataset.csv", index=False)

In [42]:
print("Final row count:", len(final_df))
print("Unique test count:", final_df["test_name"].nunique())

Final row count: 16
Unique test count: 16
